# Temporal Aggregation of EMIT Mineralogy vs. Ground Alunite Geochemistry

> This notebook is part of a series presented at the **[Workshop on Remote Sensing and Critical Mineral Discovery](https://atmos.utah.edu/critical_minerals_imaging_workshop/index.php)** presented at the University of Utah on September 29, 2026.

Authors: K. Dana Chadwick<sup>1</sup>, Philip G. Brodrick<sup>1</sup>, Erik A. Bolch<sup>2</sup>, Rupsesh Shrestha<sup>3</sup>, and Noah J. Christensen<sup>4</sup>

1. NASA Jet Propulsion Laboratory, California Institute of Technology.
2. KBR Inc., contractor to the USGS Earth Observation Research and Science Center, NASA Land Processes Distributed Active Archive Center.
3. Oak Ridge National Laboratory Distributed Active Archive Center.
4. Utah Geological Survey.

## Summary 

Visible-to-shortwave-infrared (VSWIR) imaging spectroscopy measures reflected sunlight across many adjacent wavelength bands. In this notebook, we will use the `earthaccess` python library to search for and access EMIT observations acquired over **Southern Pine Valley**, in the southern Wah Wah Range of Beaver County Utah. We will then aggregate the EMIT L2B Mineral Identification, Band Depth and Uncertainty product ([EMITL2BMIN.001](https://doi.org/10.5067/EMIT/EMITL2BMIN.001)) across those observations and compare the results against surface geochemistry data in a small region with an alunite exposure and geochemical samples provided by the Utah Geological Survey (UGS).

### Background

A single EMIT scene is one instantaneous look at the surface. During the mineral identification process using the Tetracorder expert system, identifications can be affected by things that have nothing to do with the geology: illumination geometry on that pass, thin cirrus, residual atmospheric correction error, seasonal dry vegetation, sensor noise on that particular detector column. So a mineral identified in one scene and missed in the next reports on the observation, not necessarily on the surface mineral composition. Temporal aggregation helps address this. EMIT revisits opportunistically from the ISS, and since mid-2022 it has built up repeat coverage of most of the mid-latitudes. If we classify every clear scene over an area and ask, per pixel, "in how many of its valid observations was this pixel called alunite?", then the scene-specific noise averages down and what survives is the persistent, mappable mineralogy. That persistence raster is a stronger exploration product than any single scene, and it is also a quantity (a fraction from 0 to 1) rather than a label, which is what makes it comparable with a continuous ground measurement. That ground measurement is what makes this AOI valuable: the Utah Geological Survey has 138 surface samples inside the AOI with major-oxide chemistry, and therefore two independent stoichiometric estimates of alunite abundance: one from K₂O and one from SO₃. Section 6 brings those samples together with the aggregate over the same ground, and screens them first, because alunite carries both potassium and sulfate and neither estimate is reliable without the other.

#### Geologic setting: a lithophile critical-metal system in evolved rhyolite

The Southern Pine Valley AOI (southern Wah Wah Range, Beaver County, UT ~ 38.12° N, 113.61° W) sits on the Pioche mineral belt / Blue Ribbon lineament. The AOI exposes Cambrian through Devonian carbonates, the Oligocene Needles Range Group ash-flow tuffs, and the Miocene Blawn Formation rhyolites, and it is the Blawn Formation that carries the critical-metal story. These are highly-evolved, A-type rhyolites (the ~22.1 Ma Red Beryl Rhyolite and ~18.3 Ma Tetons Rhyolite; Barkoff, 2022): textbook lithophile / critical-metal magmas that concentrate Be, Li, Mo, Sn, W, REE, U and F, carry accessory fluorite plus REE phases (cerianite-Ce, allanite, monazite, xenotime), and are named for their red beryl (bixbite). Historic work (Lindsey & Osmonson, 1978, USGS OFR 78-114) reported anomalous U, Sn (cassiterite up to ~3,000 ppm), Mo (up to ~150 ppm), Be, and Li (up to ~230 ppm), and notably analysed for gold and found none at surface. This is a lithophile suite in evolved felsic rock, comparable to the Spor Mountain, UT beryllium belt and the Climax, CO Mo system, not a gold–silver epithermal district.

#### The alteration system: a spectrally mappable pH gradient

The reason spectroscopy works here is the alteration zonation, which is a map of the mineralizing fluid's pH (Lindsey & Osmonson, 1978):

- an acidic, advanced-argillic cap (alunite + kaolinite + iron oxide + silicified breccia,
  developed in the rhyolites), the classic alunite lithocap;
- grading into near-neutral to alkaline alteration (montmorillonite + illite + fluorite)
  around intrusive topaz rhyolite (past uranium + fluorspar production at the Staats mine).

Alunite **KAl<sub>3</sub>(SO<sub>4</sub>)<sub>2</sub>(OH)<sub>6</sub>** marks the acidic core, and it is what this notebook maps: a broad Al-OH absorption at ~2.16–2.17 µm (with a secondary feature near 2.32 µm) that EMIT's Tetracorder retrieval keys on. The white-mica halo around that core is the subject of notebook 2.

> References (in `references/`): Lindsey & Osmonson (1978) USGS OFR 78-114; Barkoff (2022) UNLV
> PhD, DOI 10.34917/35777457; Portela et al. (2025) Ore Geology Reviews 182, 106673;
> UGS Open-File Report 671 (`data/ofr-671/`).

## Contents

- [1 Setup](#section-1)
- [2 Geological context and study area](#section-2)
    - [2.1 Define the regional AOI and alunite study area](#section-2-1)
    - [2.2 Visualize the geological context](#section-2-2)
    - [2.3 Describe the geologic units](#section-2-3)
- [3 Search and retain EMIT observations](#section-3)
    - [3.1 Authenticate and understand the products](#section-3-1)
    - [3.2 Search for the EMIT collections](#section-3-2)
    - [3.3 Search for mineralogy granules](#section-3-3)
    - [3.4 Retain scenes with full coverage](#section-3-4)
    - [3.5 Visualize retained footprints](#section-3-5)
    - [3.6 Find matching pixel-quality masks](#section-3-6)
- [4 Prepare and classify the scenes](#section-4)
    - [4.1 Group mineral mixtures into indicator classes](#section-4-1)
    - [4.2 Read, mask, and subset the products](#section-4-2)
    - [4.3 Define the per-scene classifier](#section-4-3)
    - [4.4 Process all retained scenes](#section-4-4)
    - [4.5 Inspect individual scenes](#section-4-5)
- [5 Align and aggregate observations](#section-5)
    - [5.1 Align scenes to a common grid](#section-5-1)
    - [5.2 Calculate temporal summaries](#section-5-2)
    - [5.3 Map modal mineral classes](#section-5-3)
    - [5.4 Map observation support, persistence, and band depth](#section-5-4)
- [6 Compare with UGS surface geochemistry](#section-6)
    - [6.1 Load and screen the samples](#section-6-1)
    - [6.2 Compare sample locations and mean band depth](#section-6-2)
- [7 Interpretation and next steps](#section-7)
    - [7.1 Extensions](#section-7-1)
    - [7.2 Save the scene selection and settings](#section-7-2)
    - [7.3 Data citations](#section-7-3)


## Learning objectives

- List the EMIT collections in NASA's CMR and pin the collection and version used.
- Search for EMIT mineralogy granules and matching quality masks over the alunite study area.
- Retain all scenes with full study-area coverage, without ranking or mosaicking.
- Group spectral-library entries into five indicator classes and apply consistent quality thresholds.
- Align scene grids and calculate mineral modes, valid-observation counts, alunite persistence, and mean alunite band depth.
- Compare the aggregate with UGS sample estimates while recognizing the different spatial scales and limits of the comparison.

## Prerequisites

A [NASA Earthdata account](https://urs.earthdata.nasa.gov/home) and the [workshop Python environment](../../../setup/setup_instructions.md) are required. Run this notebook with its directory as the working directory so the supplied files are available under `data/`.


<a id="section-1"></a>

## 1. Setup

We use EMIT mineral-identification products from multiple acquisitions to map recurring alteration-mineral detections in the alunite study area. Earthaccess provides access to the remote files, GeoPandas handles vector boundaries, and xarray holds the aligned raster observations. An Earthdata Login account is required to read the EMIT files.

Source granules are streamed rather than saved by a custom download cache. Rerunning the processing cells can transfer the data again. The number of scenes is determined by the cloud-cover and full-coverage criteria; there is no fixed scene limit.


In [4]:
from contextlib import closing
from pathlib import Path
from urllib.parse import urlparse

import earthaccess
import geopandas as gpd
import holoviews as hv
import hvplot.pandas
import hvplot.xarray
import numpy as np
import pandas as pd
import xarray as xr
from shapely.geometry import box

from vitals import emit_tools as et

DATA_DIR = Path("data")
hv.extension("bokeh")


<a id="section-2"></a>

## 2. Geological context and study area

The regional AOI provides geological context for the smaller alunite study area. Searching, raster processing, and sample screening use the smaller study-area rectangle.


<a id="section-2-1"></a>

### 2.1 Define the regional AOI and alunite study area

The regional AOI covers Southern Pine Valley in the southern Wah Wah Range, Beaver County, Utah, and provides context for the geological map. The EMIT search, raster processing, and geochemical comparison use the smaller alunite study area. We use the bounding rectangle of `Alunite_zoom.kmz` consistently for searching, scene-coverage checks, image subsetting, and sample selection.


In [2]:
# Regional boundary for the geological context map.
aoi_gdf = gpd.read_file(DATA_DIR / "AOI" / "AOI.shp").to_crs(4326)
regional_bbox = tuple(float(v) for v in aoi_gdf.total_bounds)

# Smaller study-area rectangle for searching, raster analysis, and sample comparison.
alunite_gdf = gpd.read_file(DATA_DIR / "AOI" / "Alunite_zoom.kmz").to_crs(4326)
study_bbox = tuple(float(v) for v in alunite_gdf.total_bounds)
study_poly = box(*study_bbox)
study_area_gdf = gpd.GeoDataFrame(geometry=[study_poly], crs=4326)

pd.DataFrame(
    [regional_bbox, study_bbox],
    columns=["west", "south", "east", "north"],
    index=["Regional context", "Alunite search and study area"],
)


,west,south,east,north
Regional context,-113.737629,38.016881,-113.474612,38.227452
Alunite search and study area,-113.675708,38.038942,-113.633664,38.069785


<a id="section-2-2"></a>

### 2.2 Visualize the geological context

The UGS Southern Pine Valley geologic map (Open-File Report 671) shows the mapped rock units surrounding the study area. We clip the map to the regional AOI and display the alunite study-area boundary within it. The table below lists the units that intersect the smaller study area, with their names, ages, and stratigraphic groupings.

**Figure 1 — Geological context:** mapped rock units, the regional boundary, and the alunite study-area rectangle.


In [7]:
GEO_DIR = DATA_DIR / "ofr-671"
geo = gpd.read_file(
    GEO_DIR
    / "OFR-671DM_SouthernPineValleyArea_Shapefiles"
    / "SouthernPineValleyArea_GeologicUnits.shp"
).to_crs(aoi_gdf.crs)

# GeoPandas clipping needs matching CRSs; hvPlot handles the map projection.
geo_aoi = gpd.clip(geo, aoi_gdf)

geology_map = geo_aoi.hvplot.polygons(
    geo=True, tiles="EsriWorldTopo", c="Grouping", cmap="tab20",
    fill_alpha=0.65, line_color="#4d4d4d", line_width=0.3,
    hover_cols=["UnitSymbol", "Grouping"], legend="right",
)
aoi_outline = aoi_gdf.hvplot.polygons(
    geo=True, fill_alpha=0, line_color="red", line_width=2,
    label="Regional AOI", hover=False,
)
study_outline = study_area_gdf.hvplot.polygons(
    geo=True, fill_alpha=0, line_color="cyan", line_width=2,
    line_dash="dashed", label="Alunite study area", hover=False,
)

geology_figure = (geology_map * aoi_outline * study_outline).opts(
    title="Southern Pine Valley: geological context",
    width=750, height=620, padding=0.06,
    xaxis=None, yaxis=None,
)
geology_figure


:Overlay
   .WMTS.I                      :WMTS   [Longitude,Latitude]
   .Polygons.I                  :Polygons   [Longitude,Latitude]   (Grouping,UnitSymbol)
   .Polygons.Regional_AOI       :Polygons   [Longitude,Latitude]
   .Polygons.Alunite_study_area :Polygons   [Longitude,Latitude]

<a id="section-2-3"></a>

### 2.3 Describe the geologic units

List the names, ages, and stratigraphic groupings of units intersecting the alunite study area. The table uses the mapped intersections rather than prespecifying which formation must occur there.


In [8]:
geo_attr = pd.read_csv(
    GEO_DIR / "OFR-671DM_SouthernPineValley_GeologicUnit_AttributeTable.csv",
    encoding="latin-1",
)

# Select descriptions using actual intersections with the alunite study area.
geo_study = gpd.clip(geo, study_area_gdf)
units_present = geo_study["UnitSymbol"].dropna().unique()
tbl = (
    geo_attr.loc[
        geo_attr["UnitSymbol"].isin(units_present),
        ["UnitSymbol", "UnitName", "Age", "Grouping"],
    ]
    .sort_values("UnitSymbol")
    .reset_index(drop=True)
)
tbl


,UnitSymbol,UnitName,Age,Grouping
0,Qa,Alluvium,Quaternary,Alluvial deposits
1,Ti,Isom Formation,Oligocene,Isom Formation
2,Tl,Lund Formation,Oligocene,Needles Range Group
3,Tsr,Rhyolite member of Steamboat Mountain Formation,Miocene,Steamboat Mountain Formation
4,Tt,"Rhyolite tuffs and related clastic deposits, u...",Miocene,Miocene volcanic and volcaniclastic rocks
5,Tv,"Volcanic rocks, undivided",Miocene,Miocene volcanic rocks
6,Twi,Intracaldera member of Wah Wah Springs Formation,Oligocene,Needles Range Group


<a id="section-3"></a>

## 3. Search and retain EMIT observations

Search for EMIT V001 mineral-identification granules intersecting the alunite study-area bounding box, filtering out scenes with reported cloud cover greater than 25%. Retain all returned granules whose catalog footprints fully cover the study area; no ranking or scene-count limit is applied. For simplicity, omit partial-coverage granules rather than mosaic adjacent granules. This can exclude an acquisition whose adjacent granules collectively cover the study area but do not cover it individually. Per-pixel quality flags are still applied during processing because catalog coverage and scene cloud cover do not guarantee usable observations at every pixel. The resulting composite is not an evenly sampled time series.


<a id="section-3-1"></a>

### 3.1 Authenticate and understand the products

Authenticate with Earthdata before reading remote files. EMIT L2BMIN V001 supplies mineral identifications and band depths, with fit information in its companion uncertainty file. This notebook retains the V002 standalone mask collection selected in V2.

| Product | Collection | Use |
| --- | --- | --- |
| Mineral identification and uncertainty | `EMITL2BMIN`, V001 | Mineral IDs, band depths, and fit values |
| Pixel-quality masks | `EMITL2AMASK`, V002 | Additional cloud, cirrus, and spacecraft screening |

Tetracorder compares reflectance spectra with spectral-library entries. The product records up to two identifications per pixel: Group 1 emphasizes the approximately 1 µm region, and Group 2 the approximately 2 µm region. A mineral ID identifies a library entry, band depth describes absorption strength, and fit describes the spectral match. These are not mineral percentages. ID 0 means no library match in that group; it is distinct from missing or quality-masked data.


In [9]:
earthaccess.login(persist=True)


<a id="section-3-2"></a>

### 3.2 Search for the EMIT collections

NASA's Common Metadata Repository (CMR) organizes data into *collections* (datasets) that contain *granules* (individual spatiotemporal files). Before searching for granules, list the collections themselves with `earthaccess.search_datasets` and the `keyword` argument. Each collection is identified by a `short_name`, a `version`, and a `concept_id` that is unique to that collection and version.

In [10]:
collections = earthaccess.search_datasets(keyword="EMIT")
print(f"Collections found: {len(collections)}")

# short_name, version, and concept_id are the fields used to target a collection in a granule search.
collections_info = pd.DataFrame(
    [
        {
            "short_name": c["umm"]["ShortName"],
            "version": c["umm"]["Version"],
            "concept_id": c["meta"]["concept-id"],
            "entry_title": c["umm"]["EntryTitle"],
        }
        for c in collections
    ]
)

# The keyword search also matches related records; keep the EMIT collections themselves.
emit_collections = (
    collections_info.loc[collections_info["short_name"].str.startswith("EMIT")]
    .sort_values(["short_name", "version"])
    .reset_index(drop=True)
)
pd.set_option("display.max_colwidth", 90)
emit_collections

Collections found: 72


,short_name,version,concept_id,entry_title
0,EMITL0,001,C2407897135-LPCLOUD,EMIT L0 Telemetry and Compressed Raw Instrument Data V001
1,EMITL1ARAW,001,C2407975601-LPCLOUD,EMIT L1A Reassembled Raw Image Cube 60 m V001
2,EMITL1BATT,001,C2408031090-LPCLOUD,EMIT L1B Corrected Spacecraft Attitude and Ephemeris V001
3,EMITL1BATT,002,C4079607609-LPCLOUD,EMIT L1B Corrected Spacecraft Attitude and Ephemeris V002
4,EMITL1BRAD,001,C2408009906-LPCLOUD,EMIT L1B At-Sensor Calibrated Radiance and Geolocation Data 60 m V001
5,EMITL1BRAD,002,C4079829720-LPCLOUD,EMIT L1B At-Sensor Calibrated Radiance and Geolocation Data 60 m V002
6,EMITL2AMASK,002,C3882545269-LPCLOUD,EMIT L2A Masks 60 m V002
7,EMITL2AMASK,003,C4279547358-LPCLOUD,EMIT L2A Masks 60 m V003
8,EMITL2ARFL,001,C2408750690-LPCLOUD,EMIT L2A Estimated Surface Reflectance and Uncertainty and Masks 60 m V001
9,EMITL2ARFL,002,C4079844428-LPCLOUD,EMIT L2A Surface Reflectance and Uncertainty 60 m V002


Several short names appear more than once because multiple versions of a collection are currently archived. In the future, after back-processing of scenes for the most recent version is completed, the older version will be removed from the archive. Since both versions are currently available, we want to explicitly pass `version` below. 

In [11]:
emit_collections.loc[emit_collections["short_name"].isin(["EMITL2BMIN", "EMITL2AMASK"])]

,short_name,version,concept_id,entry_title
6,EMITL2AMASK,002,C3882545269-LPCLOUD,EMIT L2A Masks 60 m V002
7,EMITL2AMASK,003,C4279547358-LPCLOUD,EMIT L2A Masks 60 m V003
18,EMITL2BMIN,001,C2408034484-LPCLOUD,EMIT L2B Estimated Mineral Identification and Band Depth and Uncertainty 60 m V001
19,EMITL2BMIN,002,C4079846859-LPCLOUD,EMIT L2B Estimated Mineral Identification and Band Depth and Uncertainty 60 m V002


For this notebook we will use `EMITL2BMIN` V001 and `EMITL2AMASK` V002 because the newer version processing has not yet delivered the scenes used in this notebook. 

<a id="section-3-3"></a>

### 3.3 Search for mineralogy granules

A granule is one spatiotemporal file within a collection. Use `earthaccess.search_data` to search the alunite study-area bounding box with a maximum reported scene cloud cover of 25%, and retrieve all matching granules before applying the footprint check. 

Granule searches accept several kinds of criteria:

| Dataset origin and location | Spatiotemporal parameters | Collection metadata parameters |
| :--- | :--- | :--- |
| `archive_center` | `bounding_box` | `concept_id` |
| `data_center` | `temporal` | `entry_title` |
| `daac` | `point` | `granule_name` |
| `provider` | `polygon` | `version` |
| `cloud_hosted` | `line` | `short_name` |

Note that not every collection reports cloud cover in its metadata, so `cloud_cover` is not a universal filter; EMIT L2B mineralogy does report it. The value is a whole-scene percentage, which is why per-pixel quality flags are still used  later to quantify the number of valid surface observations.

In [12]:
CLOUD_MAX = 25

emit_results = earthaccess.search_data(
    short_name="EMITL2BMIN",
    version="001",
    bounding_box=study_bbox,
    cloud_cover=(0, CLOUD_MAX),
)

print(f"Granules meeting the search criteria: {len(emit_results)}")


Granules meeting the search criteria: 14


c:\Users\ebolch\AppData\Local\miniforge3\envs\lpdaac_vitals\Lib\site-packages\earthaccess\results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


Each result is a metadata record that also carries the asset links for that granule. Inspect one to see the fields used below: the granule ID, acquisition time, and reported cloud cover drive the selection table, while `data_links()` supplies the mineral-identification and uncertainty files that Section 4.2 streams.

In [14]:
example = emit_results[0]
print("Granule: ", example["umm"]["GranuleUR"])
print("Acquired:", example["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"])
print("Cloud:   ", example["umm"]["CloudCover"], "%")
print("Assets:")
for url in example.data_links():
    print("   ", url)

Granule:  EMIT_L2B_MIN_001_20220822T192123_2223413_013
Acquired: 2022-08-22T19:21:23Z
Cloud:    23 %
Assets:
    https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/EMITL2BMIN.001/EMIT_L2B_MIN_001_20220822T192123_2223413_013/EMIT_L2B_MIN_001_20220822T192123_2223413_013.nc
    https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/EMITL2BMIN.001/EMIT_L2B_MIN_001_20220822T192123_2223413_013/EMIT_L2B_MINUNCERT_001_20220822T192123_2223413_013.nc


<a id="section-3-4"></a>

### 3.4 Retain scenes with full coverage

A bounding-box search returns granules that overlap any part of the study area. We keep only those whose individual footprints covering the entire study-area rectangle. Partially overlapping granules are omitted, even when adjacent granules could be combined to cover the area. This keeps the workflow focused on complete individual scenes and avoids a mosaicking step for simplicity. All scenes that pass this check are processed; chronological sorting only controls their display and processing order.


In [15]:
if not emit_results:
    raise ValueError("No granules meet the search criteria.")

# Retrieve Granule Geometries.
footprints = gpd.GeoDataFrame(
    {
        "granule": [r["umm"]["GranuleUR"] for r in emit_results],
        "date": [
            r["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"][:10]
            for r in emit_results
        ],
        "cloud_pct": [float(r["umm"]["CloudCover"]) for r in emit_results],
        "result": emit_results,
    },
    geometry=gpd.GeoSeries(emit_results, crs=4326),
)

# Keep every fully covering scene; sort chronologically without limiting the count.
footprints["covers_study_area"] = footprints.geometry.covers(study_poly)
picks = (
    footprints.loc[footprints["covers_study_area"]]
    .sort_values(["date", "granule"])
    .reset_index(drop=True)
)
if picks.empty:
    raise ValueError("No granules meet the cloud-cover and study-area coverage criteria.")

footprints["selected"] = footprints["granule"].isin(picks["granule"])
print(f"Retained {len(picks)} of {len(footprints)} granules with full study-area coverage.")
print(f"Omitted {len(footprints) - len(picks)} partial-coverage granules; no mosaicking is applied.")
picks[["date", "cloud_pct", "granule"]]


Retained 9 of 14 granules with full study-area coverage.
Omitted 5 partial-coverage granules; no mosaicking is applied.


,date,cloud_pct,granule
0,2023-04-20,7.0,EMIT_L2B_MIN_001_20230420T200006_2311013_012
1,2023-04-24,12.0,EMIT_L2B_MIN_001_20230424T182248_2311412_013
2,2023-06-27,8.0,EMIT_L2B_MIN_001_20230627T170626_2317811_012
3,2023-10-09,11.0,EMIT_L2B_MIN_001_20231009T165914_2328211_009
4,2023-10-16,6.0,EMIT_L2B_MIN_001_20231016T210319_2328914_012
5,2025-06-20,4.0,EMIT_L2B_MIN_001_20250620T182509_2517112_014
6,2025-08-01,4.0,EMIT_L2B_MIN_001_20250801T184807_2521312_012
7,2026-01-29,12.0,EMIT_L2B_MIN_001_20260129T191031_2602912_012
8,2026-04-15,11.0,EMIT_L2B_MIN_001_20260415T200247_2610513_018


<a id="section-3-5"></a>

### 3.5 Visualize retained footprints

Use `hvplot` to show the footprints on a map with our alunite study area.


In [17]:
selected_map = picks.hvplot.polygons(
    geo=True, tiles="EsriWorldTopo", c="granule", line_color="granule",
    cmap="Category20", fill_alpha=0.15, line_alpha=1, line_width=2,
    hover_cols=["granule", "date", "cloud_pct"], legend=False,
)
scene_coverage_figure = (selected_map * study_outline).opts(
    title="Selected EMIT scenes and alunite study area",
    width=750, height=550,
)
scene_coverage_figure


:Overlay
   .WMTS.I                      :WMTS   [Longitude,Latitude]
   .Polygons.I                  :Polygons   [Longitude,Latitude]   (granule,date,cloud_pct)
   .Polygons.Alunite_study_area :Polygons   [Longitude,Latitude]

<a id="section-3-6"></a>

### 3.6 Find EMIT L2A Mask Data

Search the EMIT L2A Mask v002 collection using the same alunite study area, then match results with our EMIT L2B Mineral results using the filenames which include the scene acquisition timestamp. For this search we can skip the cloud_cover argument since we'll be ensuring our resulting scenes match our previous search.

In [18]:
def asset_url(result, product):
    """Find one netCDF asset by its product filename marker."""
    urls = [u for u in result.data_links() if product in Path(urlparse(u).path).name and u.endswith(".nc")]
    if len(urls) != 1:
        raise ValueError(f"Expected one {product} asset; found {len(urls)}.")
    return urls[0]


def scene_key(result):
    """Timestamp, orbit, and scene suffix shared by companion asset filenames."""
    url = next(u for u in result.data_links() if u.endswith(".nc"))
    return "_".join(Path(urlparse(url).path).stem.split("_")[-3:])


MASK_COLLECTION = "EMITL2AMASK"
MASK_VERSION = "002"
mask_results = earthaccess.search_data(
    short_name=MASK_COLLECTION, version=MASK_VERSION,
    bounding_box=study_bbox,
    temporal=(picks["date"].min(), picks["date"].max()),
    count=-1,
)
mask_by_scene = {scene_key(r): r for r in mask_results}
missing = [scene_key(r) for r in picks["result"] if scene_key(r) not in mask_by_scene]
if missing:
    raise ValueError(f"No matching quality-mask granule for: {missing}")
print(f"Matched masks for all {len(picks)} retained mineral granules.")


Matched masks for all 9 retained mineral granules.


c:\Users\ebolch\AppData\Local\miniforge3\envs\lpdaac_vitals\Lib\site-packages\earthaccess\results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


<a id="section-4"></a>

## 4. Prepare and classify the scenes

Prepare each retained scene with the same mineral grouping, quality flags, and thresholds. Keep three layers per acquisition: indicator class, usable-observation mask, and detection band depth.


<a id="section-4-1"></a>

### 4.1 Group mineral mixtures into indicator classes

The supplied mineral-grouping table links EMIT library IDs to spectral-library names. We group those names into five tutorial classes: **alunite, pyrophyllite, kaolinite, dickite, and white mica**. 

#TODO - The ordered rules below preserve the notebook's treatment of mixtures: alunite takes precedence, followed by dickite and kaolinite; muscovite–pyrophyllite mixtures are assigned to white mica. These are analysis categories, not measurements of mineral abundance or fluid pH. Library index 0 means "no identification" in the L2B_MIN product but is not itself a row in the supplied table, so it is added explicitly and carried through the same lookup rather than left as an unchecked default.

The displayed table lists only the alunite variants and mixtures. The full lookup remains available internally for all five classes. Class 0 means no qualifying indicator detection; only the separate observation mask identifies unavailable pixels.


In [19]:
INDICATOR_ORDER = ["Alunite", "Pyrophyllite", "Kaolinite", "Dickite", "White mica"]
INDICATOR_COLORS = ["#d7191c", "#fdae61", "#ffffbf", "#abd9e9", "#2c7bb6"]
ALUNITE_CLASS = 1


def classify_name(name):
    name = str(name).lower()
    if "alunite" in name:
        return "Alunite"
    if "dickite" in name:
        return "Dickite"
    if "kaolin" in name:
        return "Kaolinite"
    if "pyrophyl" in name and "musc" not in name:
        return "Pyrophyllite"
    if any(term in name for term in ("muscovite", "illite", "sericite")):
        return "White mica"
    return None


matrix = pd.read_csv(DATA_DIR / "mineral_grouping_matrix_20230503.csv")
matrix["indicator"] = matrix["Name"].map(classify_name)

# Index 0 is reserved by the product for "no identification" and has no row in the supplied
# table; add it explicitly so it is visible in the reference table rather than only implied by
# the lookup array's default fill value below.
no_match = pd.DataFrame([{"Index": 0, "Name": "No_Match", "indicator": None}])
matrix = (
    pd.concat([no_match, matrix], ignore_index=True)
    .sort_values("Index")
    .reset_index(drop=True)
)
matrix["class_id"] = matrix["indicator"].map(
    {name: i for i, name in enumerate(INDICATOR_ORDER, start=1)}
).fillna(0).astype("int16")

# Lookup table: EMIT spectral-library Index -> indicator class. classify_scene reads this as
# id_to_class[mineral_id] instead of testing membership in each class's ID list per pixel.
id_to_class = np.zeros(int(matrix["Index"].max()) + 1, dtype="int16")
id_to_class[matrix["Index"].to_numpy()] = matrix["class_id"].to_numpy()

# Display only the spectral-library entries assigned to alunite.
matrix.loc[matrix["indicator"] == "Alunite", ["Index", "Name"]].reset_index(drop=True)


,Index,Name
0,109,Alunite GDS97 K Syn (150C) W2R4Na
1,110,Alunite GDS96 K Syn (250C) W2R4Na
2,111,Alunite RES-2 K Syn (450C) W2R4Na
3,112,Alunite GDS95 Na Syn (150C) W2R4Na
4,113,Alunite RES-4 Na Syn (300C) W2R4Na
5,114,Alunite RES-3 Na Syn (450C) W2R4Na
6,115,Alunite GDS82 Na82 W1R1Bb
7,116,Alunite+Dickite MV99-6-26b W1R1Fc
8,117,Alunite GDS83 Na63 W1R1Bb
9,118,Alunite RES-9 Summitv (400C) W2R4Nb


<a id="section-4-2"></a>

### 4.2 Read, mask, and subset the products

We open remote files with Earthaccess, apply the selected cloud, cirrus, and spacecraft flags, subset to the alunite study area, and orthorectify the data onto geographic coordinates. Each subset is loaded into memory before its remote file is closed. Streaming avoids maintaining local granule files, although the underlying readers may still transfer substantial portions of each file.

Use the Cloud, Cirrus, and Spacecraft flags (bands 0, 1, and 3). Their names are printed when reading each mask. The stored-fit threshold of 0.20 preserves the original test of doubled fit ≥ 0.40. These thresholds are analysis choices, not calibrated probabilities.


In [20]:
CLOUD_FLAG_BANDS = [0, 1, 9]   # 0 Cloud, 1 Cirrus, 9 Spec-Tf-Cloud Flag (this index will change in Mask v003)
BAND_DEPTH_MIN = 0.02
# Same numerical threshold as multiplying the stored fit by 2 and comparing to 0.40.
FIT_MIN_STORED = 0.20


def read_quality_mask(url):
    with closing(earthaccess.open([url], provider="LPCLOUD")[0]) as file:
        return et.quality_mask(file, CLOUD_FLAG_BANDS)


def read_study_subset(url, quality_mask):
    with closing(earthaccess.open([url], provider="LPCLOUD")[0]) as file:
        raw = et.emit_xarray(file, ortho=False, qmask=quality_mask)
        try:
            subset = et.spatial_subset(raw, study_area_gdf)
            return et.ortho_xr(subset).load()
        finally:
            raw.close()


<a id="section-4-3"></a>

### 4.3 Define the per-scene classifier

For each pixel, retain indicator identifications that meet the band-depth and fit thresholds. If both mineral groups supply qualifying indicators, retain the one with the greater band depth. Return three layers: the selected indicator class, a usable-observation mask, and the selected detection's band depth. Class zero means no indicator passed the rules; the observation mask distinguishes that outcome from missing or masked data.

A usable observation with no match still contributes to the observation count. Where both groups have qualifying indicators, the greater band depth wins; equal depths retain Group 1.


In [21]:
def classify_scene(result):
    mask_result = mask_by_scene[scene_key(result)]
    quality_mask = read_quality_mask(asset_url(mask_result, "_L2A_MASK_"))
    minerals = read_study_subset(asset_url(result, "_L2B_MIN_"), quality_mask)
    uncertainty = read_study_subset(asset_url(result, "_L2B_MINUNCERT_"), quality_mask)

    # Companion products must refer to the same output pixels.
    minerals, uncertainty = xr.align(minerals, uncertainty, join="exact")
    shape = minerals["group_1_band_depth"].shape
    classes = np.zeros(shape, dtype="int16")
    best_depth = np.zeros(shape, dtype="float32")
    observed = np.zeros(shape, dtype=bool)

    for group in (1, 2):
        ids = minerals[f"group_{group}_mineral_id"].values
        depth = minerals[f"group_{group}_band_depth"].values
        fit = uncertainty[f"group_{group}_fit"].values

        # Exclude missing, fractional, and out-of-range IDs before indexing the lookup.
        valid_id = np.isfinite(ids) & (ids >= 0) & (ids < id_to_class.size)
        valid_id &= ids == np.floor(ids)
        safe_ids = np.where(valid_id, ids, 0).astype("int64")
        indicator_class = id_to_class[safe_ids]

        # Include valid zero-depth observations in the persistence denominator.
        usable = np.isfinite(depth) & (depth >= 0)
        observed |= usable
        passes = (
            usable & (indicator_class > 0)
            & np.isfinite(fit) & (depth >= BAND_DEPTH_MIN) & (fit >= FIT_MIN_STORED)
        )

        # Keep the deeper of the two groups' qualifying detections; ties keep group 1.
        keep = passes & (depth > best_depth)
        classes[keep] = indicator_class[keep]
        best_depth[keep] = depth[keep]

    dims = ("latitude", "longitude")
    return xr.Dataset(
        {
            "mineral_class": (dims, classes),
            "observed": (dims, observed),
            "band_depth": (dims, np.where(classes > 0, best_depth, np.nan)),
        },
        coords={"latitude": minerals.latitude, "longitude": minerals.longitude},
    )


<a id="section-4-4"></a>

### 4.4 Process all retained scenes

Apply the same preparation and classification steps to every selected acquisition. Keep only the three small study-area layers needed for aggregation. Scene IDs and acquisition dates remain attached to the results so individual observations can be inspected.

A processing error stops execution for inspection rather than silently omitting an acquisition.


In [22]:
scenes = []
for row in picks.itertuples(index=False):
    print(f"Processing {row.date}: {row.granule}")
    scene = classify_scene(row.result)
    if not scene.observed.any().item():
        raise ValueError(f"No usable study-area pixels in {row.granule}.")
    scene = scene.expand_dims(scene=[row.granule])
    scene = scene.assign_coords(
        date=("scene", [row.date]),
        cloud_pct=("scene", [row.cloud_pct]),
    )
    scenes.append(scene)

print(f"Processed {len(scenes)} scenes.")


Processing 2023-04-20: EMIT_L2B_MIN_001_20230420T200006_2311013_012


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Flags used: ['Cloud Flag' 'Cirrus Flag' 'SpecTf-Cloud Flag']


...

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Processed 9 scenes.


<a id="section-4-5"></a>

### 4.5 Inspect individual scenes

**Figure 3 — Individual-scene mineral classes:** compare up to eight acquisitions on their original orthorectified grids before temporal aggregation. Every retained scene contributes to the later aggregate. Transparent pixels include both missing observations and pixels with no qualifying indicator detection; the observation-count map distinguishes areas with limited data support.


In [27]:
def class_map(data, title):
    return data.where(data > 0).hvplot.image(
        x="longitude", y="latitude", geo=True, tiles="EsriImagery",
        cmap=INDICATOR_COLORS, clim=(0.5, 5.5),
        width=375, height=330, title=title,
    ).opts("Image", color_levels=5, cticks=list(enumerate(INDICATOR_ORDER, start=1)), xrotation=45)


scene_panel_figure = hv.Layout([
    class_map(scene.mineral_class.isel(scene=0, drop=True), str(scene.date.values[0]))
    for scene in scenes[:8]
]).cols(2)
scene_panel_figure


:Layout
   .Overlay.I    :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (mineral_class)
   .Overlay.II   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (mineral_class)
   .Overlay.III  :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (mineral_class)
   .Overlay.IV   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (mineral_class)
   .Overlay.V    :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (mineral_class)
   .Overlay.VI   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (mineral_class)
   .Overlay.VII  :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (mineral_class)
   .Overlay.VIII :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (mineral_class)

<a id="section-5"></a>

## 5. Align and aggregate observations

Combine the classified acquisitions on a common study-area grid to summarize recurring detections and their observational support.


<a id="section-5-1"></a>

### 5.1 Align scenes to a common grid

Scene grids may have slightly different origins so we use the first selected scene as the reference and align the other scenes using nearest-neighbor resampling, and include a half-pixel tolerance along each coordinate axis. This preserves our categories and ensures selected pixels originate from a location close to their position in the common grid. Locations without a nearby source pixel remain unobserved. The reference grid comes from the first chronologically retained scene, so changing the selected dates will also change the output grid.


In [30]:
target = scenes[0]
half_lat = float(abs(target.latitude.diff("latitude")).median()) / 2
half_lon = float(abs(target.longitude.diff("longitude")).median()) / 2
fill_values = {"mineral_class": 0, "observed": False, "band_depth": np.nan}

aligned = []
for scene in scenes:
    scene = scene.reindex(
        latitude=target.latitude, method="nearest",
        tolerance=half_lat, fill_value=fill_values,
    )
    scene = scene.reindex(
        longitude=target.longitude, method="nearest",
        tolerance=half_lon, fill_value=fill_values,
    )
    aligned.append(scene)

stack = xr.concat(aligned, dim="scene", join="exact")
print(dict(stack.sizes))
stack


{'scene': 9, 'latitude': 57, 'longitude': 79}


<xarray.Dataset> Size: 285kB
Dimensions:        (scene: 9, latitude: 57, longitude: 79)
Coordinates:
  * scene          (scene) object 72B 'EMIT_L2B_MIN_001_20230420T200006_23110...
    date           (scene) <U10 360B '2023-04-20' '2023-04-24' ... '2026-04-15'
    cloud_pct      (scene) float64 72B 7.0 12.0 8.0 11.0 6.0 4.0 4.0 12.0 11.0
  * latitude       (latitude) float64 456B 38.07 38.07 38.07 ... 38.04 38.04
  * longitude      (longitude) float64 632B -113.7 -113.7 ... -113.6 -113.6
    spatial_ref    int64 8B 0
Data variables:
    mineral_class  (scene, latitude, longitude) int16 81kB 0 0 0 0 0 ... 0 0 0 0
    observed       (scene, latitude, longitude) bool 41kB True True ... True
    band_depth     (scene, latitude, longitude) float32 162kB nan nan ... nan

<a id="section-5-2"></a>

### 5.2 Calculate temporal summaries

For each pixel, calculate the number of usable observations, the two most frequent detected indicator classes, the fraction of usable observations classified as alunite, and the mean band depth of the alunite detections. The band-depth summary is shown where alunite is the most frequent detected indicator. Three usable observations are required for the reported quantitative summaries.

Modal classes exclude class zero and therefore describe the most frequent detected indicator, even if detections are uncommon. Ties are resolved by class order, with alunite first. The second mode describes variation among identifications; it does not by itself establish mineral mixing. Persistence measures repeated detection, and band depth measures spectral absorption strength rather than mineral percentage.


In [31]:
MIN_OBS = 3

# Count votes for each detected indicator, excluding class zero.
class_index = pd.Index(range(1, 6), name="class_id")
votes = xr.concat(
    [(stack.mineral_class == i).sum("scene") for i in class_index],
    dim=class_index,
)
agree1 = votes.max("class_id")
mode1 = votes.idxmax("class_id").where(agree1 > 0, 0)
remaining_votes = votes.where(votes.class_id != mode1, 0)
mode2 = remaining_votes.idxmax("class_id").where(remaining_votes.max("class_id") > 0, 0)

# Normalize detections by usable observations at each pixel.
n_obs = stack.observed.sum("scene")
enough_observations = n_obs >= MIN_OBS
alunite_detected = stack.mineral_class == ALUNITE_CLASS
alunite_count = alunite_detected.sum("scene")
persistence = (alunite_count / n_obs.where(n_obs > 0)).where(enough_observations)
mean_depth = stack.band_depth.where(alunite_detected).mean("scene")
mean_depth = mean_depth.where(enough_observations & (mode1 == ALUNITE_CLASS))

aggregate = xr.Dataset({
    "n_obs": n_obs,
    "mode1": mode1,
    "mode2": mode2,
    "mode1_votes": agree1,
    "alunite_persistence": persistence,
    "mean_alunite_band_depth": mean_depth,
})
aggregate


<xarray.Dataset> Size: 199kB
Dimensions:                  (latitude: 57, longitude: 79)
Coordinates:
  * latitude                 (latitude) float64 456B 38.07 38.07 ... 38.04 38.04
  * longitude                (longitude) float64 632B -113.7 -113.7 ... -113.6
    spatial_ref              int64 8B 0
Data variables:
    n_obs                    (latitude, longitude) int64 36kB 9 9 9 9 ... 8 8 8
    mode1                    (latitude, longitude) int64 36kB 0 0 0 0 ... 0 0 0
    mode2                    (latitude, longitude) int64 36kB 0 0 0 0 ... 0 0 0
    mode1_votes              (latitude, longitude) int64 36kB 0 0 0 0 ... 0 0 0
    alunite_persistence      (latitude, longitude) float64 36kB 0.0 0.0 ... 0.0
    mean_alunite_band_depth  (latitude, longitude) float32 18kB nan nan ... nan

<a id="section-5-3"></a>

### 5.3 Map modal mineral classes

**Figure 4 — Modal mineral classes:** show the most frequent and second-most frequent detected indicator over the alunite study area, masking pixels with fewer than three usable observations. This retains the modal-map comparison from the outline while removing the duplicate regional panels, since raster processing now covers only the study area.


In [32]:
supported = aggregate.n_obs >= MIN_OBS
modal_class_figure = (
    class_map(aggregate.mode1.where(supported), "Most frequent detected indicator")
    + class_map(aggregate.mode2.where(supported), "Second most frequent indicator")
).cols(2)
modal_class_figure


:Layout
   .Overlay.I  :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (mode1)
   .Overlay.II :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (mode2)

<a id="section-5-4"></a>

### 5.4 Map observation support, persistence, and band depth

**Figure 4b — Quantitative summaries:** show usable-observation counts, the fraction of those observations classified as alunite, and mean alunite detection band depth. Interpret persistence together with observation counts. Band depth is shown only where alunite is the most frequent detected indicator and the minimum observation count is met.


In [33]:
map_options = dict(
    x="longitude", y="latitude", geo=True, tiles="EsriImagery",
    width=350, height=330,
)

observation_map = aggregate.n_obs.hvplot.image(
    **map_options, cmap="viridis", clim=(0, len(scenes)),
    title="Usable observations", clabel="Observations",
)
persistence_map = aggregate.alunite_persistence.hvplot.image(
    **map_options, cmap="magma", clim=(0, 1),
    title="Alunite detection fraction", clabel="Fraction",
)
depth_map = aggregate.mean_alunite_band_depth.hvplot.image(
    **map_options, cmap="Greys_r",
    title="Mean alunite band depth", clabel="Band depth",
)

summary_figure = (observation_map + persistence_map + depth_map).cols(3)
summary_figure


:Layout
   .Overlay.I   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (n_obs)
   .Overlay.II  :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (alunite_persistence)
   .Overlay.III :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (mean_alunite_band_depth)

<a id="section-6"></a>

## 6. Compare with UGS surface geochemistry

The UGS sample table contains coordinates and alunite estimates derived from K₂O and SO₃. Retain samples within the alunite study area and identify those with numeric estimates of at least 3% from both methods. Values recorded as text limits or negative sentinels are excluded from the numeric comparison. Other samples include estimates below the threshold, disagreements between methods, and incomplete results; they are not all confirmed absences of alunite.

We compare the spatial patterns of the sample estimates and EMIT band depth. Hand samples and EMIT pixels represent different spatial scales, and the geochemical values are estimates from bulk chemistry. This map is a qualitative comparison, not a measurement of pixel-level accuracy.


<a id="section-6-1"></a>

### 6.1 Load and screen the samples

Retain samples inside the same study-area rectangle and apply the two-estimate threshold. Keep other or incomplete estimates as a separate category, without interpreting every such sample as an alunite absence. Text limits such as `>30` are excluded from the numeric comparison under the existing policy; they are not zero values.


In [34]:
gchem = pd.read_csv(
    DATA_DIR / "geochem" / "alunite_geochem_subset_2026_09_09.csv",
    encoding="utf-8-sig",
)
gchem.columns = gchem.columns.str.strip()
gchem_gdf = gpd.GeoDataFrame(
    gchem,
    geometry=gpd.points_from_xy(gchem["Longitude"], gchem["Latitude"]),
    crs=4326,
)
gchem_gdf = gchem_gdf.loc[gchem_gdf.geometry.intersects(study_poly)].copy()

ALUN_COL = "pct_alunite_from_K2O"
ALUN_SO3 = "pct_alunite_from_SO3"
ALUN_MIN_PCT = 3.0

for column in (ALUN_COL, ALUN_SO3):
    values = pd.to_numeric(gchem_gdf[column], errors="coerce")
    gchem_gdf[column] = values.where(values >= 0)

gchem_gdf["alun_both"] = (
    (gchem_gdf[ALUN_COL] >= ALUN_MIN_PCT)
    & (gchem_gdf[ALUN_SO3] >= ALUN_MIN_PCT)
)
gchem_gdf["sample_group"] = np.where(
    gchem_gdf.alun_both, "Both estimates meet threshold", "Other or incomplete estimates",
)
gchem_gdf["sample_group"].value_counts()


sample_group
Both estimates meet threshold    76
Other or incomplete estimates    28
Name: count, dtype: int64

<a id="section-6-2"></a>

### 6.2 Compare sample locations and mean band depth

Two quantities share one map, so the figure carries two color scales. The grayscale raster underneath is the mean EMIT alunite band depth where alunite is the most frequent detected indicator and at least three usable observations are available; it is stretched between the 5th and 95th percentiles of the mapped pixels, because band depth has no natural 0-to-1 range and fixed limits would either saturate the core of the exposure or flatten its weaker margins. The magma-colored circles on top are the K₂O-based estimates for samples meeting both geochemical thresholds, stretched from zero to the 98th percentile of those samples so that a few high assays do not compress the rest of the range into one shade. Cyan crosses mark the other samples; they carry no color, because an estimate the second route does not corroborate is not worth placing on a color scale. They still matter, since the map is only informative if the ground is also negative where EMIT aggregated no alunite.

Read the map for coincidence of pattern rather than point-by-point agreement, and note where the raster is blank because alunite is not modal or observations are too few.

**Figure 5 — UGS samples over mean EMIT alunite band depth.** The K₂O-based estimate is the mapped quantity, following the UGS XRPD comparison described in the original notebook, which found closer agreement for samples corroborated by both estimates. The SO₃-based estimate provides the second screening criterion; neither estimate alone establishes a mineral abundance at EMIT pixel scale.

In [35]:
accepted = gchem_gdf.loc[gchem_gdf.alun_both]
other = gchem_gdf.loc[~gchem_gdf.alun_both]

# Percentile stretches: band depth has no natural 0-1 range, and a few high assays would otherwise
# compress the sample colors into a single shade.
depth_lo, depth_hi = np.nanpercentile(aggregate.mean_alunite_band_depth, [5, 95])
sample_clim = (0.0, float(np.nanpercentile(accepted[ALUN_COL], 98)))

comparison = aggregate.mean_alunite_band_depth.hvplot.image(
    x="longitude", y="latitude", geo=True, tiles="EsriImagery",
    cmap="Greys_r", clim=(float(depth_lo), float(depth_hi)), colorbar=True,
    clabel=f"EMIT mean alunite band depth ({len(scenes)}-scene aggregate)",
    alpha=0.85, width=760, height=620,
)

if not accepted.empty:
    # Bokeh keeps one colorbar per side, so the sample scale goes on the left of the map.
    comparison *= accepted.hvplot.points(
        geo=True, c=ALUN_COL, cmap="magma", clim=sample_clim,
        colorbar=True, clabel="UGS alunite from K₂O (%)",
        size=90, line_color="black",
        hover_cols=[ALUN_COL, ALUN_SO3],
        label=f"Both estimates meet threshold (n={len(accepted)})",
    ).opts(colorbar_position="left")
if not other.empty:
    comparison *= other.hvplot.points(
        geo=True, color="cyan", marker="x", size=70, line_width=2,
        hover_cols=[ALUN_COL, ALUN_SO3],
        label=f"Other or incomplete estimates (n={len(other)})",
    )

comparison = comparison.opts(
    title="UGS samples and mean EMIT alunite band depth",
    legend_position="top_right",
    legend_opts={"background_fill_alpha": 0.85, "label_text_font_size": "9pt"},
)
comparison

:Overlay
   .WMTS.I                                                                              :WMTS   [Longitude,Latitude]
   .Image.I                                                                             :Image   [longitude,latitude]   (mean_alunite_band_depth)
   .Points.Both_estimates_meet_threshold_left_parenthesis_n_equals_76_right_parenthesis :Points   [Longitude,Latitude]   (pct_alunite_from_K2O,pct_alunite_from_SO3)
   .Points.Other_or_incomplete_estimates_left_parenthesis_n_equals_28_right_parenthesis :Points   [Longitude,Latitude]   (pct_alunite_from_K2O,pct_alunite_from_SO3)

<a id="section-7"></a>

## 7. Interpretation and next steps

This workflow summarizes repeated mineral detections within the alunite study area. Observation counts describe data support, modal classes describe the most frequent detected indicators, persistence describes the frequency of alunite identification, and mean band depth describes the strength of the retained alunite absorptions.

The maps support comparison with UGS sample locations and estimates. They do not directly measure mineral abundance, metal concentration, or fluid pH. Variation among acquisitions can reflect surface conditions, masking, registration, mixtures, and classification thresholds. Interpret the second modal class as variation among identifications rather than proof of a particular cause.

Record the retained granule IDs and thresholds when comparing results across runs. Only individually complete catalog footprints are included; partial scenes and acquisitions requiring adjacent-granule mosaics are omitted. Repeating an unrestricted search later can change the observations as the archive grows.


<a id="section-7-1"></a>

### 7.1 Extensions

- Compare results under different cloud, band-depth, fit, and observation-count thresholds.
- Compare seasonal subsets when enough acquisitions are available; inspect changes in observation support as well as detections.
- Continue to `02_white_mica_absorption_centers.ipynb` to explore white-mica absorption positions and comparisons with airborne reflectance.
- Inspect original library matches and mixtures before assigning a geological cause to variation among mineral classes.


<a id="section-7-2"></a>

### 7.2 Save the scene selection and settings

Save the small scene-selection table and analysis settings for comparison across runs. These files do not cache the remote granules. The interactive figures are displayed in the notebook and are not exported to files.


In [ ]:
picks[["date", "cloud_pct", "granule"]].to_csv(
    DATA_DIR / "alunite_selected_scenes.csv", index=False,
)
pd.Series({
    "product_version": "001",
    "mask_collection": MASK_COLLECTION,
    "mask_version": MASK_VERSION,
    "cloud_max": CLOUD_MAX,
    "band_depth_min": BAND_DEPTH_MIN,
    "fit_min_stored": FIT_MIN_STORED,
    "min_observations": MIN_OBS,
    "cloud_flag_bands": CLOUD_FLAG_BANDS,
    "study_bbox": study_bbox,
    "coverage_policy": "Retain all individually covering granules; omit partial scenes; no mosaicking",
}).to_json(DATA_DIR / "alunite_analysis_settings.json", indent=2)


<a id="section-7-3"></a>

### 7.3 Data citations

- EMIT L2B Mineralogy: Green, R. et al. (2024). EMIT L2B Estimated Mineral Identification and
  Band Depth and Uncertainty 60 m V001. NASA LP DAAC. DOI 10.5067/EMIT/EMITL2BMIN.001
- EMIT L2A Reflectance & Mask: DOI 10.5067/EMIT/EMITL2ARFL.001
- Utah Geological Survey, Open-File Report 671 (geologic map) and the accompanying surface
  geochemical dataset.
- Lindsey, D.A. & Osmonson, L.M. (1978). Mineral potential of altered rocks near Blawn Mountain,
  Wah Wah Range, Utah. USGS Open-File Report 78-114.
- Barkoff, D.W. (2022). PhD dissertation, University of Nevada Las Vegas. DOI 10.34917/35777457
- Portela, B. et al. (2025). Ore Geology Reviews 182, 106673.



## Contact Info:  

Email: LPDAAC@usgs.gov  
Voice: +1-866-573-3222  
Organization: Land Processes Distributed Active Archive Center (LP DAAC)¹  
Website: <https://www.earthdata.nasa.gov/centers/lp-daac>  

¹Work performed under USGS contract 140G0126D0001 for NASA contract NNG14HH33I. 